<a href="https://colab.research.google.com/github/Brahmareddy-Ambavarapu/Personalized-Hybrid-Recommendation-System/blob/main/Personalized_Hybrid_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# STEP 1: LOAD AND PREPARE DATA
# ============================================================

import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from collections import defaultdict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

df = pd.read_csv("/content/personalized_recommendation_dataset.csv")

df["Timestamp"] = pd.to_datetime(df["Timestamp"])

print("Dataset shape:", df.shape)
print(df.head())

Device: cpu
Dataset shape: (150000, 8)
     User_ID    Item_ID     Category  Rating  Timestamp   Price Platform  \
0   User_913    Item_52       Movies     2.0 2023-05-15  369.55      Web   
1  User_3457    Item_66  Electronics     1.4 2023-08-19  255.15      Web   
2  User_1629  Item_1467       Sports     2.7 2024-03-27  296.69      Web   
3  User_3463   Item_697       Movies     1.6 2023-12-03   55.59   Tablet   
4  User_2941  Item_1736        Games     3.4 2023-02-06  366.22      Web   

        Location  
0         Africa  
1         Africa  
2         Europe  
3  North America  
4  South America  


In [2]:
# ============================================================
# STEP 2: TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

def temporal_split(df):
    data = df.sort_values(["User_ID", "Timestamp"]).copy()

    train_parts = []
    val_parts = []
    test_parts = []

    for user_id, user_data in data.groupby("User_ID"):

        user_data = user_data.sort_values("Timestamp")

        # Users with fewer than 3 interactions
        # remain in training.
        if len(user_data) < 3:
            train_parts.append(user_data)
            continue

        test_parts.append(user_data.iloc[-1:])
        val_parts.append(user_data.iloc[-2:-1])
        train_parts.append(user_data.iloc[:-2])

    train_df = pd.concat(train_parts).reset_index(drop=True)
    val_df = pd.concat(val_parts).reset_index(drop=True) if val_parts else pd.DataFrame()
    test_df = pd.concat(test_parts).reset_index(drop=True) if test_parts else pd.DataFrame()

    return train_df, val_df, test_df


train_df, val_df, test_df = temporal_split(df)

print("\nTemporal split")
print("----------------------------")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nDate ranges")

print(
    "Train:",
    train_df["Timestamp"].min(),
    "→",
    train_df["Timestamp"].max()
)

print(
    "Validation:",
    val_df["Timestamp"].min(),
    "→",
    val_df["Timestamp"].max()
)

print(
    "Test:",
    test_df["Timestamp"].min(),
    "→",
    test_df["Timestamp"].max()
)


Temporal split
----------------------------
Train: (140000, 8)
Validation: (5000, 8)
Test: (5000, 8)

Date ranges
Train: 2022-12-24 00:00:00 → 2024-12-22 00:00:00
Validation: 2024-03-26 00:00:00 → 2024-12-23 00:00:00
Test: 2024-05-25 00:00:00 → 2024-12-23 00:00:00


In [3]:
# ============================================================
# STEP 3: CREATE REAL INTERACTION GROUND TRUTH
# ============================================================

# Unique users and items
all_users = sorted(df["User_ID"].unique())
all_items = sorted(df["Item_ID"].unique())

user_to_idx = {u: i for i, u in enumerate(all_users)}
item_to_idx = {i: j for j, i in enumerate(all_items)}

idx_to_user = {v: k for k, v in user_to_idx.items()}
idx_to_item = {v: k for k, v in item_to_idx.items()}

n_users = len(all_users)
n_items = len(all_items)

print("Users:", n_users)
print("Items:", n_items)


# ------------------------------------------------------------
# Positive interactions
# ------------------------------------------------------------

train_positive = defaultdict(set)
val_positive = defaultdict(set)
test_positive = defaultdict(set)

for _, row in train_df.iterrows():
    train_positive[user_to_idx[row["User_ID"]]].add(
        item_to_idx[row["Item_ID"]]
    )

for _, row in val_df.iterrows():
    val_positive[user_to_idx[row["User_ID"]]].add(
        item_to_idx[row["Item_ID"]]
    )

for _, row in test_df.iterrows():
    test_positive[user_to_idx[row["User_ID"]]].add(
        item_to_idx[row["Item_ID"]]
    )


print("\nExample user histories:")

for user_id in list(train_positive.keys())[:3]:

    print(
        idx_to_user[user_id],
        "train interactions:",
        len(train_positive[user_id]),
        "| validation:",
        len(val_positive.get(user_id, set())),
        "| test:",
        len(test_positive.get(user_id, set()))
    )

Users: 5000
Items: 2000

Example user histories:
User_1 train interactions: 25 | validation: 1 | test: 1
User_10 train interactions: 36 | validation: 1 | test: 1
User_100 train interactions: 26 | validation: 1 | test: 1


In [4]:
# ============================================================
# STEP 4: ITEM CONTENT FEATURES
# ============================================================

content_df = df.copy()

# Numerical price
price_scaler = StandardScaler()

content_df["Price_scaled"] = price_scaler.fit_transform(
    content_df[["Price"]]
)

# One-hot categorical features
content_matrix = pd.get_dummies(
    content_df[
        ["Item_ID", "Category", "Platform", "Location"]
    ],
    columns=["Category", "Platform", "Location"]
)

# Keep one row per item
item_content = (
    content_matrix
    .groupby("Item_ID")
    .first()
)

# Add price
item_price = (
    content_df
    .groupby("Item_ID")["Price_scaled"]
    .mean()
    .rename("Price_scaled")
)

item_content = item_content.join(item_price)

# Reindex to our item mapping
item_content = item_content.reindex(all_items)

item_content = item_content.fillna(0)

content_features = item_content.values.astype(np.float32)

print("Content feature shape:", content_features.shape)

Content feature shape: (2000, 20)


In [5]:
# ============================================================
# STEP 5: POPULARITY BASELINE
# ============================================================

item_popularity = (
    train_df
    .groupby("Item_ID")
    .size()
    .sort_values(ascending=False)
)

popular_items = [
    item_to_idx[item]
    for item in item_popularity.index
]

print("Most popular items:")

for item_idx in popular_items[:10]:
    print(idx_to_item[item_idx])

Most popular items:
Item_276
Item_286
Item_1617
Item_1938
Item_507
Item_343
Item_595
Item_234
Item_1905
Item_1638


In [6]:
# ============================================================
# STEP 6: CONTENT-BASED RECOMMENDER
# ============================================================

content_similarity = cosine_similarity(
    content_features
)

def content_recommend(
    user_id,
    k=10,
    exclude_seen=True
):

    seen_items = train_positive.get(user_id, set())

    user_history = list(seen_items)

    if len(user_history) == 0:
        return popular_items[:k]

    # User profile = average content of interacted items
    user_profile = content_features[user_history].mean(axis=0)

    # Similarity between user profile and every item
    scores = cosine_similarity(
        user_profile.reshape(1, -1),
        content_features
    ).flatten()

    if exclude_seen:
        scores[list(seen_items)] = -np.inf

    top_items = np.argsort(scores)[::-1][:k]

    return top_items.tolist()

In [7]:
# ============================================================
# STEP 7: COLLABORATIVE FILTERING BASELINE
# ============================================================

class MatrixFactorization(nn.Module):

    def __init__(
        self,
        n_users,
        n_items,
        embedding_dim=32
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            n_items,
            embedding_dim
        )

        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)

        nn.init.normal_(
            self.user_embedding.weight,
            std=0.01
        )

        nn.init.normal_(
            self.item_embedding.weight,
            std=0.01
        )

    def forward(self, users, items):

        user_emb = self.user_embedding(users)
        item_emb = self.item_embedding(items)

        dot = (user_emb * item_emb).sum(dim=1)

        bias = (
            self.user_bias(users).squeeze()
            +
            self.item_bias(items).squeeze()
        )

        return dot + bias

In [8]:
# ============================================================
# TRAIN MATRIX FACTORIZATION
# ============================================================

def train_mf(
    train_positive,
    n_users,
    n_items,
    embedding_dim=32,
    epochs=20,
    lr=0.005,
    negative_samples=2
):

    model = MatrixFactorization(
        n_users,
        n_items,
        embedding_dim
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-5
    )

    criterion = nn.BCEWithLogitsLoss()

    users = []
    items = []
    labels = []

    for user, positive_items in train_positive.items():

        for item in positive_items:

            users.append(user)
            items.append(item)
            labels.append(1)

            for _ in range(negative_samples):

                negative_item = np.random.randint(n_items)

                while negative_item in positive_items:
                    negative_item = np.random.randint(n_items)

                users.append(user)
                items.append(negative_item)
                labels.append(0)

    users = torch.LongTensor(users).to(device)
    items = torch.LongTensor(items).to(device)
    labels = torch.FloatTensor(labels).to(device)

    for epoch in range(epochs):

        model.train()

        optimizer.zero_grad()

        scores = model(users, items)

        loss = criterion(scores, labels)

        loss.backward()

        optimizer.step()

        if epoch % 5 == 0:
            print(
                f"MF Epoch {epoch:02d} | "
                f"Loss: {loss.item():.4f}"
            )

    return model


mf_model = train_mf(
    train_positive,
    n_users,
    n_items
)

MF Epoch 00 | Loss: 0.8968
MF Epoch 05 | Loss: 0.8862
MF Epoch 10 | Loss: 0.8729
MF Epoch 15 | Loss: 0.8542


In [9]:
# ============================================================
# STEP 8: IMPROVED HYBRID NEURAL RECOMMENDER
# ============================================================

class HybridRecommender(nn.Module):

    def __init__(
        self,
        n_users,
        n_items,
        content_dim,
        embedding_dim=32
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            n_items,
            embedding_dim
        )

        self.content_projection = nn.Sequential(
            nn.Linear(content_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.20)
        )

        self.mlp = nn.Sequential(

            nn.Linear(
                embedding_dim * 2 + 32,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.30),

            nn.Linear(64, 32),

            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Linear(32, 1)
        )

    def forward(
        self,
        users,
        items,
        content
    ):

        user_emb = self.user_embedding(users)

        item_emb = self.item_embedding(items)

        content_emb = self.content_projection(content)

        combined = torch.cat(
            [
                user_emb,
                item_emb,
                content_emb
            ],
            dim=1
        )

        return self.mlp(combined).squeeze(1)

In [10]:
# ============================================================
# STEP 9: TRAIN HYBRID MODEL WITH VALIDATION
# ============================================================

def create_training_examples(
    train_positive,
    n_items,
    negative_samples=2
):

    users = []
    items = []
    labels = []

    for user, positive_items in train_positive.items():

        for item in positive_items:

            users.append(user)
            items.append(item)
            labels.append(1)

            for _ in range(negative_samples):

                neg = np.random.randint(n_items)

                while neg in positive_items:
                    neg = np.random.randint(n_items)

                users.append(user)
                items.append(neg)
                labels.append(0)

    return (
        np.array(users),
        np.array(items),
        np.array(labels)
    )


train_users_arr, train_items_arr, train_labels = \
    create_training_examples(
        train_positive,
        n_items
    )


# Validation examples
val_users_arr = []
val_items_arr = []
val_labels = []

for user, positives in val_positive.items():

    seen = train_positive.get(user, set())

    for item in positives:

        val_users_arr.append(user)
        val_items_arr.append(item)
        val_labels.append(1)

        for _ in range(2):

            neg = np.random.randint(n_items)

            while (
                neg in seen
                or neg in positives
            ):
                neg = np.random.randint(n_items)

            val_users_arr.append(user)
            val_items_arr.append(neg)
            val_labels.append(0)


val_users_arr = np.array(val_users_arr)
val_items_arr = np.array(val_items_arr)
val_labels = np.array(val_labels)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

hybrid_model = HybridRecommender(
    n_users,
    n_items,
    content_features.shape[1]
).to(device)

optimizer = optim.Adam(
    hybrid_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

criterion = nn.BCEWithLogitsLoss()

best_val_loss = float("inf")
best_state = None

patience = 5
patience_counter = 0

batch_size = 512


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

for epoch in range(50):

    hybrid_model.train()

    permutation = np.random.permutation(
        len(train_labels)
    )

    total_loss = 0

    for start in range(
        0,
        len(permutation),
        batch_size
    ):

        idx = permutation[
            start:start + batch_size
        ]

        users = torch.LongTensor(
            train_users_arr[idx]
        ).to(device)

        items = torch.LongTensor(
            train_items_arr[idx]
        ).to(device)

        labels = torch.FloatTensor(
            train_labels[idx]
        ).to(device)

        content = torch.FloatTensor(
            content_features[items.cpu().numpy()]
        ).to(device)

        optimizer.zero_grad()

        scores = hybrid_model(
            users,
            items,
            content
        )

        loss = criterion(
            scores,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    hybrid_model.eval()

    with torch.no_grad():

        val_users_tensor = torch.LongTensor(
            val_users_arr
        ).to(device)

        val_items_tensor = torch.LongTensor(
            val_items_arr
        ).to(device)

        val_content_tensor = torch.FloatTensor(
            content_features[val_items_arr]
        ).to(device)

        val_labels_tensor = torch.FloatTensor(
            val_labels
        ).to(device)

        val_scores = hybrid_model(
            val_users_tensor,
            val_items_tensor,
            val_content_tensor
        )

        val_loss = criterion(
            val_scores,
            val_labels_tensor
        ).item()

    train_loss = total_loss / max(
        1,
        len(permutation) // batch_size
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = {
            k: v.cpu().clone()
            for k, v in hybrid_model.state_dict().items()
        }

        patience_counter = 0

    else:

        patience_counter += 1

        if patience_counter >= patience:

            print("Early stopping triggered.")

            break


# Restore best model
hybrid_model.load_state_dict(best_state)

hybrid_model = hybrid_model.to(device)

print("Best validation loss:", best_val_loss)

Epoch 01 | Train Loss: 0.6402 | Val Loss: 0.6368
Epoch 02 | Train Loss: 0.6380 | Val Loss: 0.6366
Epoch 03 | Train Loss: 0.6377 | Val Loss: 0.6365
Epoch 04 | Train Loss: 0.6376 | Val Loss: 0.6366
Epoch 05 | Train Loss: 0.6375 | Val Loss: 0.6366
Epoch 06 | Train Loss: 0.6375 | Val Loss: 0.6366
Epoch 07 | Train Loss: 0.6375 | Val Loss: 0.6365
Epoch 08 | Train Loss: 0.6374 | Val Loss: 0.6365
Epoch 09 | Train Loss: 0.6374 | Val Loss: 0.6365
Epoch 10 | Train Loss: 0.6374 | Val Loss: 0.6365
Epoch 11 | Train Loss: 0.6374 | Val Loss: 0.6365
Epoch 12 | Train Loss: 0.6374 | Val Loss: 0.6366
Epoch 13 | Train Loss: 0.6373 | Val Loss: 0.6366
Epoch 14 | Train Loss: 0.6374 | Val Loss: 0.6365
Epoch 15 | Train Loss: 0.6373 | Val Loss: 0.6365
Epoch 16 | Train Loss: 0.6374 | Val Loss: 0.6365
Early stopping triggered.
Best validation loss: 0.6365143656730652


In [11]:
# ============================================================
# STEP 10: RECOMMENDATION METRICS
# ============================================================

def ndcg_at_k(recommended, relevant, k):

    recommended = recommended[:k]

    dcg = 0.0

    for rank, item in enumerate(recommended):

        if item in relevant:

            dcg += 1 / np.log2(rank + 2)

    ideal_hits = min(
        len(relevant),
        k
    )

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg


def evaluate_top_k(
    recommendation_function,
    test_positive,
    k=10
):

    precisions = []
    recalls = []
    ndcgs = []

    for user, relevant_items in test_positive.items():

        if len(relevant_items) == 0:
            continue

        recommendations = recommendation_function(
            user,
            k
        )

        recommendations = recommendations[:k]

        hits = len(
            set(recommendations)
            &
            relevant_items
        )

        precision = hits / k

        recall = hits / len(relevant_items)

        ndcg = ndcg_at_k(
            recommendations,
            relevant_items,
            k
        )

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        f"Precision@{k}": np.mean(precisions),
        f"Recall@{k}": np.mean(recalls),
        f"NDCG@{k}": np.mean(ndcgs)
    }

In [12]:
# ============================================================
# POPULARITY RECOMMENDER
# ============================================================

def popularity_recommend(user_id, k=10):

    seen = train_positive.get(user_id, set())

    recommendations = [
        item
        for item in popular_items
        if item not in seen
    ]

    return recommendations[:k]

In [13]:
# ============================================================
# COLLABORATIVE FILTERING RECOMMENDER
# ============================================================

def collaborative_recommend(user_id, k=10):

    seen = train_positive.get(user_id, set())

    all_item_ids = np.arange(n_items)

    user_tensor = torch.LongTensor(
        [user_id] * n_items
    ).to(device)

    item_tensor = torch.LongTensor(
        all_item_ids
    ).to(device)

    mf_model.eval()

    with torch.no_grad():

        scores = mf_model(
            user_tensor,
            item_tensor
        ).cpu().numpy()

    scores[list(seen)] = -np.inf

    top_items = np.argsort(scores)[::-1][:k]

    return top_items.tolist()

In [14]:
# ============================================================
# HYBRID NEURAL RECOMMENDER
# ============================================================

def hybrid_recommend(user_id, k=10):

    seen = train_positive.get(user_id, set())

    candidates = np.arange(n_items)

    user_array = np.full(
        n_items,
        user_id
    )

    user_tensor = torch.LongTensor(
        user_array
    ).to(device)

    item_tensor = torch.LongTensor(
        candidates
    ).to(device)

    content_tensor = torch.FloatTensor(
        content_features
    ).to(device)

    hybrid_model.eval()

    with torch.no_grad():

        scores = hybrid_model(
            user_tensor,
            item_tensor,
            content_tensor
        ).cpu().numpy()

    scores[list(seen)] = -np.inf

    top_items = np.argsort(scores)[::-1][:k]

    return top_items.tolist()

In [15]:
# ============================================================
# STEP 14: BASELINE COMPARISON
# ============================================================

print("\nEvaluating Popularity...")
pop_results = evaluate_top_k(
    popularity_recommend,
    test_positive,
    k=10
)

print("\nEvaluating Content-Based...")
content_results = evaluate_top_k(
    content_recommend,
    test_positive,
    k=10
)

print("\nEvaluating Collaborative Filtering...")
cf_results = evaluate_top_k(
    collaborative_recommend,
    test_positive,
    k=10
)

print("\nEvaluating Hybrid Neural...")
hybrid_results = evaluate_top_k(
    hybrid_recommend,
    test_positive,
    k=10
)


results = pd.DataFrame(
    [
        pop_results,
        content_results,
        cf_results,
        hybrid_results
    ],
    index=[
        "Popularity",
        "Content-Based",
        "Collaborative Filtering",
        "Hybrid Neural"
    ]
)

print("\n==============================")
print("MODEL COMPARISON")
print("==============================")

display(results)


Evaluating Popularity...

Evaluating Content-Based...

Evaluating Collaborative Filtering...

Evaluating Hybrid Neural...

MODEL COMPARISON


,Precision@10,Recall@10,NDCG@10
Popularity,0.00052,0.0052,0.002498
Content-Based,0.00050,0.0050,0.002324
Collaborative Filtering,0.00060,0.0060,0.002907
Hybrid Neural,0.00046,0.0046,0.001726


In [16]:
# ============================================================
# STEP 15: EVALUATE AT K=5
# ============================================================

results_k5 = pd.DataFrame(
    [
        evaluate_top_k(
            popularity_recommend,
            test_positive,
            k=5
        ),
        evaluate_top_k(
            content_recommend,
            test_positive,
            k=5
        ),
        evaluate_top_k(
            collaborative_recommend,
            test_positive,
            k=5
        ),
        evaluate_top_k(
            hybrid_recommend,
            test_positive,
            k=5
        )
    ],
    index=[
        "Popularity",
        "Content-Based",
        "Collaborative Filtering",
        "Hybrid Neural"
    ]
)

print("\n==============================")
print("K=5 RESULTS")
print("==============================")

display(results_k5)


K=5 RESULTS


,Precision@5,Recall@5,NDCG@5
Popularity,0.00064,0.0032,0.001847
Content-Based,0.00060,0.0030,0.001671
Collaborative Filtering,0.00084,0.0042,0.002335
Hybrid Neural,0.00028,0.0014,0.000698


In [17]:
# ============================================================
# STEP 16: ABLATION MODEL
# ============================================================

class AblationRecommender(nn.Module):

    def __init__(
        self,
        n_users,
        n_items,
        content_dim,
        use_content=True,
        use_reward=False,
        embedding_dim=32
    ):

        super().__init__()

        self.use_content = use_content
        self.use_reward = use_reward

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            n_items,
            embedding_dim
        )

        input_dim = embedding_dim * 2

        if use_content:

            self.content_projection = nn.Sequential(
                nn.Linear(content_dim, 32),
                nn.ReLU(),
                nn.Dropout(0.2)
            )

            input_dim += 32

        if use_reward:

            input_dim += 1

        self.mlp = nn.Sequential(

            nn.Linear(input_dim, 64),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(64, 32),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(
        self,
        users,
        items,
        content,
        reward=None
    ):

        user_emb = self.user_embedding(users)

        item_emb = self.item_embedding(items)

        features = [
            user_emb,
            item_emb
        ]

        if self.use_content:

            content_emb = self.content_projection(
                content
            )

            features.append(content_emb)

        if self.use_reward:

            reward = reward.unsqueeze(1)

            features.append(reward)

        combined = torch.cat(
            features,
            dim=1
        )

        return self.mlp(combined).squeeze(1)

In [18]:
# ============================================================
# STEP 17: ABLATION STUDY
# ============================================================

ablation_results = []

# ------------------------------------------------------------
# Model A: Collaborative only
# ------------------------------------------------------------

ablation_results.append({
    "Model": "User + Item",
    "Description": "Collaborative embeddings only"
})


# ------------------------------------------------------------
# Model B: Collaborative + Content
# ------------------------------------------------------------

ablation_results.append({
    "Model": "User + Item + Content",
    "Description": "Collaborative + content features"
})


# ------------------------------------------------------------
# Model C: Collaborative + Content + Reward
# ------------------------------------------------------------

ablation_results.append({
    "Model": "User + Item + Content + Reward",
    "Description": "Full hybrid model"
})


ablation_summary = pd.DataFrame(
    ablation_results
)

display(ablation_summary)

,Model,Description
0,User + Item,Collaborative embeddings only
1,User + Item + Content,Collaborative + content features
2,User + Item + Content + Reward,Full hybrid model
